# 과제 — AI 거버넌스, 혼자 처음부터 끝까지  ·  과제 (TAKE-HOME)
### 주제 3: AI 거버넌스 · 4주간 배운 6차원 툴킷을 *스스로* 들이댄다

> **이 과제 한 문장:** "우크라이나·가자에서는 강사와 함께 분석했다.
> 이제 **AI 거버넌스**라는 새 코퍼스에 너 혼자 `diplo_analysis` 툴킷을 들고 들어가,
> *누가 침묵했는지* 찾아내고, *당사자성이 낮은 주제에서 각국의 말하기가
> 수렴하는지 발산하는지* 를 직접 검증한다."

이번 코퍼스는 **작다(약 50건)**. 작은 데이터는 그 자체로 발견이다 —
어떤 소스는 아예 **한 건도 없다**. 그 침묵을 네가 찾아내는 게 첫 임무다.

#### 🎯 학습 목표
1. 배운 툴킷(`diplo_analysis`)을 **새 주제에 스스로** 적용한다.
2. **침묵 지도**로 *구조적으로 부재한 소스* 를 발견하고 그 이유를 추론한다.
3. **당사자성(stake) 가설**을 검증한다: 고당사자성(우크라이나/가자) vs
   저당사자성(AI 거버넌스)에서 언어 패턴(상호성·동사강도·완곡어)이 다른가?
4. **v2 차원을 재해석·적용**한다: `event_naming` 을 *위협 vs 기회 프레임* 으로 다시 쓰고(Step 4a),
   *귀속·대상별태도*(blame/targeted)가 AI엔 **0건**임을 확인해 그 의미를 읽는다(Step 4b).
5. 결과를 **차트 한 장**으로 보여주고, 분석의 **한계**를 스스로 적는다.

#### ❓ 연구 질문 (recap)
- **Q1.** 같은 사건을 두고 소스(UN/한국/중국/프랑스)별로 말하기가 다른가?
- **Q2.** 주제의 **당사자성**이 언어를 바꾸는가? — 우크라이나·가자는 소스들이
  *직접 당사자*(고당사자성)지만, AI 거버넌스는 *한 발 떨어진* 글로벌 거버넌스
  (저당사자성)다. 그럼 강도·상호성·완곡어가 달라지는가?

> 💡 **진행 방식:** 셀을 위에서 아래로 하나씩 실행한다.
> `# TODO` 가 보이면 빈칸(`____`)을 직접 채우고, 바로 아래 `# CHECK` 셀을 실행해 `✅` 가 떠야 다음으로.
> `📝 서술형` 마크다운 셀은 **너의 문장으로** 채운다(채점 대상).

## Step 0 — 환경 설정
필요한 라이브러리를 설치하고, 프로젝트 폴더를 찾는다.

In [ ]:
!pip install spacy plotly pandas -q
!python -m spacy download en_core_web_sm -q

In [ ]:
# ── 프로젝트 환경 자동 설정 (Colab / 로컬 공용) ───────────────────────
# 이 셀은 모든 세션 노트북 맨 위에 동일하게 들어간다. 그냥 실행만 하면 된다.
import os, sys, json, glob

def find_project():
    """data/backup/corpus_clean.json 이 있는 program4 폴더를 찾는다."""
    cands = [".", "program4", "..", "../program4", "/content/program4",
             "/content/edu/program4", os.path.expanduser("~/program4")]
    for c in cands:
        if os.path.exists(os.path.join(c, "data", "backup", "corpus_clean.json")):
            return os.path.abspath(c)
    return None

PROJECT = find_project()
if PROJECT is None:
    print("⚠️  데이터를 찾지 못했습니다. 아래 둘 중 하나로 해결하세요:")
    print("  (A) Colab: 좌측 파일창에 program4 폴더를 통째로 업로드")
    print("  (B) Colab: !git clone <이 강의 repo 주소>  후 다시 실행")
else:
    sys.path.insert(0, PROJECT)
    os.chdir(PROJECT)
    print("✅ 프로젝트 경로:", PROJECT)


## Step 1 — 데이터 존재 확인 + AI 거버넌스 불러오기 🔍

분석 전 항상 **존재 → 개수 → 내용** 순으로 두 눈으로 확인한다(분석의 첫 윤리).
이번엔 간단히만 확인하고, `topic == "ai_governance"` 만 골라낸다.

In [ ]:
import json, os, pandas as pd
docs_all = json.load(open(os.path.join(PROJECT,"data","backup","corpus_clean.json"), encoding="utf-8"))
df = pd.DataFrame(docs_all)
print("전체 코퍼스:", df.shape, "| 주제:", dict(df["topic"].value_counts()))

In [ ]:
# TODO: topic 이 "ai_governance" 인 행만 골라 ai 변수에 담아라.
#       힌트: df[df["topic"] == "____"]
ai = df[df["topic"] == "______________"].copy()
print("AI 거버넌스 문서:", len(ai), "건")
ai.groupby("source").size()

In [ ]:
# CHECK Step1 — AI 거버넌스가 제대로 골라졌는가 (작은 코퍼스: ~50건)
try:
    assert set(ai["topic"].unique()) == {"ai_governance"}, "필터가 잘못됐다"
    assert 40 <= len(ai) <= 60, "건수가 예상 범위를 벗어났다"
    print("✅ PASS — AI 거버넌스", len(ai), "건 확보 (작은 코퍼스다)")
except Exception as e:
    print("❌ FAIL —", e, '\n힌트: df[df["topic"] == "ai_governance"]')

<details><summary>💡 힌트</summary>

빈칸은 주제 이름 문자열이다. `df["topic"]` 컬럼에 어떤 값들이 있는지 위 셀에서 봤다.
정답 코드는 정답본 노트북에 있다.
</details>

## Step 2 — 침묵 지도: 어느 소스가 *구조적으로 부재* 하는가? 🤫

> **이번 과제의 첫 발견.** 우크라이나(S1)에서 배웠듯 **말하지 않는 것도 데이터**다.
> AI 거버넌스 8개 사건(ai01~ai08)에 대해 **사건 × 소스** 매트릭스를 만들어,
> *어떤 소스가 거의 / 아예 말하지 않았는지* 를 네가 직접 찾아낸다.

In [ ]:
# TODO: silence_map 으로 AI 거버넌스 침묵 지도를 만들어라.
#   - silence_map(docs, events) 는 사건×소스 커버리지(0=침묵)를 돌려준다.
#   - docs 인자에는 ai 의 레코드 리스트, events 인자에는 ai_governance 사건 목록.
#   힌트: from diplo_analysis import silence_map / from scrapers.events import events_for
from diplo_analysis import silence_map
from scrapers.events import events_for

ai_docs = ai.to_dict("records")
sm = silence_map(ai_docs, events_for("________________"))   # ← 빈칸: 어떤 topic?
cov = pd.DataFrame(sm)[["event_id","event_name","UN","KR","CN","FR","silent_sources"]]
cov

In [ ]:
# TODO: 소스별 총 문서 수를 세어, '아예 0건인 소스'를 찾아라.
#       힌트: cov[["UN","KR","CN","FR"]].sum() 으로 소스별 합계를 구한다.
src_totals = cov[["UN","KR","CN","FR"]].____()    # ← 빈칸: 합계 메서드
print("소스별 총 문서 수:\n", src_totals)
absent = [s for s in ["UN","KR","CN","FR"] if src_totals[s] == 0]
print("\n구조적으로 부재(0건)한 소스:", absent)

In [ ]:
# CHECK Step2 — 부재한 소스를 제대로 찾았는가
try:
    src_totals = cov[["UN","KR","CN","FR"]].sum()
    absent = [s for s in ["UN","KR","CN","FR"] if src_totals[s] == 0]
    assert absent == ["FR"], f"부재 소스가 예상과 다르다: {absent}"
    print("✅ PASS — 프랑스(FR)가 AI 거버넌스에서 단 한 건도 없다(구조적 침묵).")
    print("   이게 왜 그런지는 Step 5(a) 서술형에서 설명한다.")
except Exception as e:
    print("❌ FAIL —", e, '\n힌트: 소스별 합계가 0인 곳을 찾아라')

<details><summary>💡 힌트</summary>

- 첫 칸은 `events_for("ai_governance")`.
- 둘째 칸은 `.sum()` — 컬럼별 합계를 낸다.
- 합계가 0인 소스가 바로 "한 건도 안 낸" = **구조적으로 부재한** 소스다.
- *왜* 부재할까는 곧 서술형에서 묻는다. (이 코퍼스가 어디서 긁혔는지 떠올려 보라.)
</details>

## Step 3 — `analyze_document` 전 문서 분석 → 소스 비교표

이제 툴킷의 `analyze_document` 를 **AI 거버넌스 전 문서**에 돌려 6차원을 뽑고,
**소스별 평균표**를 만든다. (프랑스는 데이터가 없으니 표에 UN/KR/CN 만 나온다.)

> ⚙️ S4에서 배운 교훈을 잊지 말 것: `verb_strength_max`(원시 최댓값)는 **문서 길이에
> 오염**된다. 그래서 아래에서 **밀도(`verb_density` = 강도동사수/단어수×1000)** 도 함께 만든다.
> 결론은 *밀도* 위에서 말한다.

In [ ]:
from diplo_analysis import get_nlp, analyze_document
nlp = get_nlp()                       # spaCy 1회 로드
rows = [analyze_document(d, nlp) for d in ai_docs]   # 50건 전부 분석 (수 초)
ana = pd.DataFrame(rows)
# 원문 텍스트를 붙여 길이/단어수 계산 (S4의 길이 편향 교훈)
ana = ana.merge(ai[["id","text","title","url","lang"]], on="id", how="left")
ana["n_words"] = ana["text"].str.split().str.len()
print("분석 결과표:", ana.shape)
ana[["source","event_id","directness_index","verb_strength_max","mutuality_index","hedging_density"]].head(6)

In [ ]:
# TODO: S4에서 배운 '밀도 정규화'를 그대로 적용하라.
#   verb_density = 강도동사 개수 / 단어수 × 1000
#   힌트: ana["n_strength_verbs"] 를 ana["n_words"] 로 나누고 1000 을 곱한다.
ana["verb_density"] = ana["________________"] / ana["n_words"] * 1000   # ← 빈칸

# 소스별 평균표 (밀도 포함). FR은 데이터가 없어 자동으로 빠진다.
cols = ["directness_index","verb_strength_max","verb_density",
        "mutuality_index","hedging_density","subject_first_person"]
agg = ana.groupby("______")[cols].mean().round(2)   # ← 빈칸: 무엇으로 묶나?
agg = agg.reindex([s for s in ["UN","KR","CN","FR"] if s in agg.index])
agg

In [ ]:
# CHECK Step3 — 소스 비교표가 알려진 실제 결과와 맞는가 (소수 오차 허용)
try:
    assert "verb_density" in ana.columns and ana["verb_density"].notna().all()
    agg = ana.groupby("source")[cols].mean().round(2)
    assert "FR" not in agg.index, "FR은 데이터가 없어야 한다"
    assert set(agg.index) == {"UN","KR","CN"}, "소스가 UN/KR/CN 셋이어야 한다"
    # 상호성: 한국·중국이 UN보다 높다 (저당사자성에서 협력 어휘 多)
    assert agg.loc["CN","mutuality_index"] > agg.loc["UN","mutuality_index"]
    assert agg.loc["KR","mutuality_index"] > agg.loc["UN","mutuality_index"]
    print("✅ PASS — 소스 비교표 완성 (UN/KR/CN, 프랑스는 부재)")
    print("   상호성: 한국 %.1f · 중국 %.1f > UN %.1f"
          % (agg.loc["KR","mutuality_index"], agg.loc["CN","mutuality_index"], agg.loc["UN","mutuality_index"]))
except Exception as e:
    print("❌ FAIL —", e, '\n힌트: 빈칸은 "n_strength_verbs" 와 "source"')

<details><summary>💡 힌트</summary>

- 첫 칸은 `"n_strength_verbs"` (S4에서 똑같이 했다 — 개수를 길이로 나눠 밀도로).
- 둘째 칸은 `"source"` — 소스별로 묶어 평균.
- 표에 행이 **3개(UN/KR/CN)** 만 나오면 정상이다. 프랑스는 데이터가 없다.
</details>

## Step 4 — 당사자성(stake) 가설 검증: 고당사자성 vs 저당사자성

> **Q2의 심장.** 우크라이나·가자에서 각국은 *직접 당사자*다(고당사자성).
> AI 거버넌스는 *한 발 떨어진* 글로벌 거버넌스다(저당사자성).
> 그럼 언어가 달라지는가? 아래 **우크라이나·가자 기준 수치(강사가 미리 계산)** 와
> 네가 방금 구한 AI 거버넌스 수치를 **나란히** 비교한다.

아래는 강사가 같은 툴킷으로 미리 계산한 **주제별 전체 평균(모든 소스 합산)** 이다.
이걸 기준선(reference)으로 삼는다.

In [ ]:
# 강사 제공 기준선 — 우크라이나/가자(고당사자성) 주제별 전체 평균.
# (같은 diplo_analysis 툴킷으로 계산한 값. 너의 AI 거버넌스 결과와 비교용.)
REFERENCE = pd.DataFrame({
    "mutuality_index":   {"ukraine": 5.74, "gaza": 6.47},
    "verb_strength_max": {"ukraine": 4.57, "gaza": 4.54},
    "verb_density":      {"ukraine": 5.52, "gaza": 5.51},
    "hedging_density":   {"ukraine": 3.28, "gaza": 3.50},
    "directness_index":  {"ukraine": 0.80, "gaza": 0.79},
})
REFERENCE

In [ ]:
# TODO: 네 AI 거버넌스 데이터(ana)의 '전체 평균'을 구해 기준선에 한 줄로 추가하라.
#   비교할 5개 지표의 평균을 내고 round(2) 한다.
#   힌트: ana[[지표들]].mean() → REFERENCE.loc["ai_governance"] = ...
metrics = ["mutuality_index","verb_strength_max","verb_density","hedging_density","directness_index"]
ai_mean = ana[metrics].____().round(2)     # ← 빈칸: 평균 메서드
compare = REFERENCE.copy()
compare.loc["ai_governance"] = ai_mean
compare

In [ ]:
# CHECK Step4 — 당사자성 대비가 드러나는가
try:
    assert "ai_governance" in compare.index, "AI 거버넌스 행이 없다"
    m = compare["mutuality_index"]
    v = compare["verb_density"]
    # 저당사자성(AI)일수록 상호성↑, 동사밀도(강경)↓ 라는 가설
    assert m["ai_governance"] > m["ukraine"] and m["ai_governance"] > m["gaza"], "상호성 가설 불일치"
    assert v["ai_governance"] < v["ukraine"] and v["ai_governance"] < v["gaza"], "동사밀도 가설 불일치"
    print("✅ PASS — 저당사자성(AI)일수록 상호성↑ 동사강도(밀도)↓ 패턴이 보인다.")
    print("   상호성: AI %.2f vs 우크라 %.2f / 가자 %.2f" % (m["ai_governance"], m["ukraine"], m["gaza"]))
    print("   동사밀도: AI %.2f vs 우크라 %.2f / 가자 %.2f" % (v["ai_governance"], v["ukraine"], v["gaza"]))
except Exception as e:
    print("❌ FAIL —", e, '\n힌트: 빈칸은 .mean()')

<details><summary>💡 힌트</summary>

빈칸은 `.mean()`. `ana[metrics].mean()` 은 각 지표의 전체 평균(시리즈)을 준다.
그걸 `compare.loc["ai_governance"]` 에 한 줄로 넣으면 3주제 비교표가 완성된다.
</details>

## Step 4a — AI 명명 프레임: 위협(threat)인가 기회(opportunity)인가? 🧭

> **v2 차원 적용 (event_naming, 재해석판).** 우크라이나·가자에서 `event_naming` 은
> *침공/학살 vs 사태/작전* 같은 **완곡명명 사다리**였다. AI 거버넌스엔 가해 사건이 없으니
> 같은 함수를 **프레임 사다리**로 다시 쓴다 — `existential risk/threat/danger`(위협계, 가중치 2~3),
> `risk/challenge/safety`(중간, 1), `opportunity/benefit/potential`(기회계, 0).
>
> 질문: **어느 소스가 AI를 "위협"으로, 어느 소스가 "기회"로 프레임하나?**
> escalation 평균(UN 0.79 / KR 0.74 / CN 0.77)은 서로 가깝다 — 평균만 보면 안 보인다.
> 그래서 **위협계 vs 기회계 단어 카운트 분포**(더 선명한 신호)를 직접 센다.

In [ ]:
# TODO: 각 문서에 event_naming(text, "ai_governance") 을 돌려 명명 단어를 모으고,
#       소스별로 '위협계(weight>=2)'와 '기회계(weight==0)' 단어 수를 합산하라.
#   힌트: from diplo_analysis import event_naming, EVENT_NAMING
#         가중치표 = EVENT_NAMING["ai_governance"]["en"]  (단어→가중치)
from diplo_analysis import event_naming, EVENT_NAMING
from collections import Counter
W = EVENT_NAMING["ai_governance"]["en"]          # {단어: 가중치}

frame_rows = []
for d in ai_docs:
    nm = event_naming(d["text"], "______________")   # ← 빈칸: 어떤 topic?
    terms = nm["naming_terms"]                        # {단어: 횟수}
    threat = sum(c for t, c in terms.items() if W[t] >= 2)   # 위협계
    middle = sum(c for t, c in terms.items() if W[t] == 1)   # 중간(risk/challenge/safety)
    chance = sum(c for t, c in terms.items() if W[t] == 0)   # 기회계
    frame_rows.append({"source": d["source"], "위협계": threat,
                       "중간": middle, "기회계": chance,
                       "escalation": nm["naming_escalation"]})
frame = pd.DataFrame(frame_rows)
# 소스별 위협계/기회계 단어 총합 + escalation 평균
frame_by_src = frame.groupby("______").agg(            # ← 빈칸: 무엇으로 묶나?
    위협계=("위협계","sum"), 중간=("중간","sum"), 기회계=("기회계","sum"),
    escalation평균=("escalation","mean")).round(2)
frame_by_src = frame_by_src.reindex([s for s in ["UN","KR","CN","FR"] if s in frame_by_src.index])
frame_by_src

In [ ]:
# CHECK Step4a — 명명 프레임 분포가 실제 결과와 맞는가
try:
    assert set(frame_by_src.index) <= {"UN","KR","CN"}, "FR은 데이터가 없다"
    # 중국은 기회계 > 위협계 (AI를 기회로 프레임)
    assert frame_by_src.loc["CN","기회계"] > frame_by_src.loc["CN","위협계"], "중국=기회 편향이 안 보인다"
    # UN은 위협계가 0이 아니다 (위협·관리 프레임을 분명히 쓴다)
    assert frame_by_src.loc["UN","위협계"] > 0, "UN 위협계가 0이면 안 된다"
    print("✅ PASS — 중국은 AI를 '기회'로(기회계 > 위협계), UN은 위협·관리 어휘를 고루 쓴다.")
    print("   중국 기회계 %d vs 위협계 %d | UN 위협계 %d 기회계 %d"
          % (frame_by_src.loc["CN","기회계"], frame_by_src.loc["CN","위협계"],
             frame_by_src.loc["UN","위협계"], frame_by_src.loc["UN","기회계"]))
except Exception as e:
    print("❌ FAIL —", e, '\n힌트: 빈칸은 "ai_governance" 와 "source"')

<details><summary>💡 힌트</summary>

- 첫 칸은 `"ai_governance"`, 둘째 칸은 `"source"`.
- `nm["naming_terms"]` 는 `{"benefit": 19, "threat": 9, ...}` 같은 *단어→횟수* 딕셔너리다.
- 가중치표 `W`(=`EVENT_NAMING["ai_governance"]["en"]`)에서 단어의 가중치를 찾아
  `>=2`면 위협계, `==0`이면 기회계로 분류한다.
- **escalation 평균은 세 소스가 거의 같다(0.7~0.8)** — 평균에 속지 말고 *분포*를 보라.
</details>

## Step 4b — 차원 적용 가능성의 *비대칭*: 왜 어떤 자(尺)는 AI에 안 들어맞나? 🧩

> **이것도 발견이다 — 버그가 아니라.** v2 툴킷엔 전쟁 분석용 차원이 둘 더 있다:
> `blame_attribution`(누구를 가해자로 지목했나)·`targeted_sentiment`(행위자별 태도 비대칭).
> 우크라이나·가자(S1·S4)에선 이 둘이 핵심이었다. **AI 거버넌스에 그대로 돌리면 어떻게 될까?**
> 직접 돌려서 결과를 *눈으로* 확인하라. (결과 자체가 Q2 당사자성 가설의 강한 증거가 된다.)

In [ ]:
# TODO: blame_attribution / targeted_sentiment 를 AI 전 문서에 돌려
#       'AI를 가해자로 지목한 문서 수'와 '대상별 태도가 측정된 문서 수'를 세어라.
#   힌트: from diplo_analysis import blame_attribution, targeted_sentiment
#         blame 은 blame_named(>0이면 지목), targeted 는 targeted_sentiment(빈 dict가 아니면 측정됨).
from diplo_analysis import blame_attribution, targeted_sentiment

n_blame = 0      # 가해자를 명시한 문서 수
n_target = 0     # 대상별 태도가 하나라도 잡힌 문서 수
for d in ai_docs:
    bl = blame_attribution(d["text"], nlp, "______________", d.get("lang","en"))  # ← 빈칸: topic
    ts = targeted_sentiment(d["text"], "ai_governance", d.get("lang","en"))
    if bl["blame_named"] > 0:
        n_blame += 1
    if ts["targeted_sentiment"]:        # 빈 dict 가 아니면
        n_target += 1
print(f"AI 코퍼스 {len(ai_docs)}건 중 → 가해자 지목 {n_blame}건 · 대상별 태도 측정 {n_target}건")

In [ ]:
# CHECK Step4b — 전쟁용 차원이 AI에선 '0건'으로 비는가 (이게 정답이다)
try:
    assert n_blame == 0, f"가해자 지목이 0이어야 하는데 {n_blame}건"
    assert n_target == 0, f"대상별 태도가 0이어야 하는데 {n_target}건"
    print("✅ PASS — blame/targeted 모두 0건. AI엔 가해자도 대립 당사자도 없다(버그 아님, 발견임).")
    print("   왜 전쟁에선 되고 AI에선 안 되는지는 서술형 (d)에서 설명한다.")
except Exception as e:
    print("❌ FAIL —", e, '\n힌트: 빈칸은 "ai_governance". AGGRESSIVE_VERBS/ACTORS 에 ai_governance 가 없다.')

<details><summary>💡 힌트</summary>

- 빈칸은 `"ai_governance"`.
- 결과는 **둘 다 0건**이 정상이다. 에러가 아니다 — 함수가 *빈 결과*를 정상적으로 돌려준다.
- 왜? `diplo_analysis` 의 `AGGRESSIVE_VERBS`·`ACTORS` 사전에 **ai_governance 키가 없다**.
  가해자(perpetrator)도, 서로 대립하는 당사자(opposing parties)도 정의돼 있지 않기 때문이다.
- 이 *0* 이 무엇을 뜻하는지는 곧 서술형 (d)에서 직접 쓴다.
</details>

> 🧪 **(참고) 부정처리는 이제 기본이다.** Step 3·4 에서 쓴 `mutuality_index`·`hedging_density` 는
> v2부터 **부정 범위(negation scope)를 자동으로 거른다**(`handle_negation=True` 기본값). 예컨대
> *"there is **no** mutual interest"* 의 'mutual'은 상호성으로 **세지 않는다**. 네가 따로 코딩할 건 없고,
> "협력 어휘 카운트에 *부정문 오탐*이 이미 빠져 있다"는 점만 알고 결과를 읽으면 된다.

## Step 5 — 시각화 (Plotly) 한 장 📊

표는 정확하지만 눈에 안 들어온다. **너의 선택**으로 차트 한 장을 그려라.
아래는 가장 무난한 예시(주제별 상호성 비교 막대)의 골격이다. 빈칸만 채우면 된다.
원하면 레이더/산점도 등 다른 형태로 바꿔도 좋다(채점은 '의도가 드러나는가'로 본다).

In [ ]:
# TODO: 주제별 상호성(mutuality) 비교 막대그래프를 완성하라.
#   x = 세 주제, y = compare["mutuality_index"] 값.
#   힌트: go.Bar(x=..., y=...) 의 y 에 어떤 열을 넣을지 생각.
import plotly.graph_objects as go
topics_ko = {"ukraine":"우크라이나(高당사자성)","gaza":"가자(高당사자성)","ai_governance":"AI거버넌스(低당사자성)"}
order = ["ukraine","gaza","ai_governance"]
fig = go.Figure(go.Bar(
    x=[topics_ko[t] for t in order],
    y=[compare.loc[t, "________________"] for t in order],   # ← 빈칸: 어떤 지표?
    marker_color=["#E45756","#F58518","#4C78A8"],
    text=[f'{compare.loc[t,"mutuality_index"]:.1f}' for t in order],
    textposition="outside",
))
fig.update_layout(
    title="당사자성이 낮을수록 상호성(협력 어휘)이 높다",
    xaxis_title="주제(당사자성)", yaxis_title="상호성 지수 (1000단어당 양면적 표현)",
    template="plotly_white", height=440,
)
fig.show()

In [ ]:
# CHECK Step5 — 차트가 그려졌고 데이터가 올바르게 들어갔는가
try:
    ys = list(fig.data[0].y)
    assert len(ys) == 3 and abs(ys[2] - compare.loc["ai_governance","mutuality_index"]) < 1e-6
    print("✅ PASS — 시각화 완성 (AI 거버넌스 막대가 가장 높다)")
except Exception as e:
    print("❌ FAIL —", e, '\n힌트: 빈칸은 "mutuality_index"')

<details><summary>💡 힌트</summary>

빈칸은 `"mutuality_index"`. 다른 차트를 그려도 되지만, 이 CHECK 셀은 위 예시 기준이다 —
다른 형태로 바꿨다면 CHECK는 건너뛰고 직접 그림이 뜨는지 확인하라.
</details>

## Step 6 — 📝 서술형 질문 (채점 대상)

아래 세 칸을 **너의 문장으로** 채운다. 정답이 하나는 아니다. 채점은
*숫자를 근거로 댔는가 / 측정의 한계를 아는가* 를 본다. 각 3~6문장.

### (a) 왜 프랑스(FR)는 AI 거버넌스에서 한 건도 없는가? 🇫🇷

침묵 지도에서 FR=0 을 찾았다. 이것이 *우발적 누락*이 아니라 *구조적 부재*라면 그 이유는?
(힌트: 이 코퍼스는 어디서 긁혔나? 프랑스의 AI 외교 자료는 어디에 사는가?)

> _여기에 답을 적으세요 (3~6문장)._
>
>

### (b) 저당사자성 주제(AI)에서 4개(실제론 3개) 소스의 언어는 *수렴* 하는가 *발산* 하는가?

Step 3 소스 비교표와 Step 4 당사자성 비교표를 근거로 답하라.
고당사자성(우크라이나/가자)과 비교해 소스 간 차이가 줄었나 늘었나? 어떤 지표에서?

> _여기에 답을 적으세요 (3~6문장)._
>
>

### (c) 이 분석의 한계 2가지

네가 한 분석을 *스스로 반증* 한다면 어디를 공격하겠는가? 구체적으로 2가지.

> _여기에 한계 2가지를 적으세요._
>
> 1.
> 2.

### (d) 왜 *귀속(blame)·대상별 태도(targeted)* 는 전쟁에선 작동하고 AI에선 0건인가? 🧩

Step 4b에서 `blame_attribution`·`targeted_sentiment` 가 AI 코퍼스에서 **둘 다 0건**임을 봤다.
우크라이나·가자(S1·S4)에서는 같은 함수가 핵심이었다. **무엇이 다른가?**
(힌트: 이 0은 코드 고장인가, 아니면 주제의 *구조*가 다른 것인가? 당사자성과 연결하라.)

> _여기에 답을 적으세요 (3~6문장). 0이 '에러'인지 '발견'인지부터 판단하라._
>
>

## (선택·가산점) Claude 의미 분석 확장 🤖

규칙 기반 점수가 *놓친* framing/미묘한 hedging을 Claude로 교차검증할 수 있다.
**API 키가 없으면 자동으로 건너뛴다(DEMO_MODE).** 키가 있을 때만 한 문서를 분석한다.

> 설계 원칙: *계산은 코드(결정론), 의미 해석은 LLM, 판단은 인간.*
> 모델 id는 `diplo_analysis.claude_semantic` 의 기본값을 따른다.

In [ ]:
# 키가 있을 때만 실행 — 없으면 조용히 DEMO_MODE 로 넘어간다 (채점 비대상, 가산점)
import os
DEMO_MODE = not os.environ.get("ANTHROPIC_API_KEY")
if DEMO_MODE:
    print("ℹ️  DEMO_MODE — ANTHROPIC_API_KEY 가 없어 Claude 확장을 건너뜁니다(정상).")
    print("   규칙 기반 분석만으로 이 과제는 100% 완료됩니다.")
else:
    import anthropic
    from diplo_analysis import claude_semantic
    client = anthropic.Anthropic()
    sample = ai_docs[0]
    result = claude_semantic(sample["text"], client)   # framing/blame/강도/상호성/미묘한 hedging
    print("Claude 의미 분석 (", sample["source"], sample["event_id"], "):")
    for k, v in result.items():
        print(f"  {k}: {v}")

## ✅ 제출 전 체크 + 회고

**제출 전 확인**
- [ ] 모든 `# CHECK` 셀에 `✅ PASS` 가 떴다 (Step 1·2·3·4·**4a·4b**·5 — 총 7개).
- [ ] Step 6 서술형 (a)(b)(c)**(d)** 를 *내 문장* 으로 채웠다.
- [ ] 차트가 한 장 이상 그려진다.
- [ ] 위에서 아래로 한 번에 다시 실행해도 에러가 없다(Runtime → Restart and run all).

**회고 (제출 안 함, 스스로)**
1. "FR=0"을 처음 봤을 때 무엇이라 해석했나? 구조적 부재임을 안 뒤 해석이 어떻게 바뀌었나?
2. 저당사자성에서 상호성이 *올라간* 게 의외였나? 왜 그럴까?
3. 이 작은 코퍼스(50건)로 기사를 쓴다면, 어떤 문장은 쓰면 안 되는가?

> 🎓 **수고했다.** 4주간 우크라이나·가자에서 함께 깎은 자(尺)를, 새 주제에 혼자 들이대
> *침묵을 발견*하고 *당사자성 가설을 검증*했다. 좋은 분석가의 마지막 덕목 — **자기 숫자를
> 의심하기** — 를 서술형 (c)에서 실천했다면, 이 과제의 진짜 목표를 달성한 것이다.